# Online Retail II: cleaning and quality checks
Merges the two yearly sheets of the *Online Retail II* dataset, drops rows without a customer ID, fixes types, fills missing descriptions from the stock code, and adds `TotalPrice` and `IsCancelled`. Also checks the quality of a small ad-conversion dataset.

**Data:** not included, see the repository README for sources.

In [ ]:
%cd /content/data

!ls

/content/data
 cleaned_online_retail_2009_2011.csv
 joined.csv
 KAG_conversion_data.csv
 online_retail_II.xlsx
'online_retail_II.xlsx - Year 2009-2010.csv'
'online_retail_II.xlsx - Year 2010-2011.csv'


In [ ]:
import pandas as pd
df1= pd.read_excel('online_retail_II.xlsx',sheet_name="Year 2009-2010")
df2= pd.read_excel('online_retail_II.xlsx',sheet_name="Year 2010-2011")
frm1=df1
frm2=df2
print(frm1)
print(frm2)


       Invoice StockCode                          Description  Quantity  \
0       489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1       489434    79323P                   PINK CHERRY LIGHTS        12   
2       489434    79323W                  WHITE CHERRY LIGHTS        12   
3       489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4       489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...        ...       ...                                  ...       ...   
525456  538171     22271                 FELTCRAFT DOLL ROSIE         2   
525457  538171     22750         FELTCRAFT PRINCESS LOLA DOLL         1   
525458  538171     22751       FELTCRAFT PRINCESS OLIVIA DOLL         1   
525459  538171     20970   PINK FLORAL FELTCRAFT SHOULDER BAG         2   
525460  538171     21931               JUMBO STORAGE BAG SUKI         2   

               InvoiceDate  Price  Customer ID         Country  
0      2009-12-01 07:45:00   6.95 

In [ ]:


# Merge them
df = pd.concat([frm1, frm2], ignore_index=True)

print("✅ Combined shape:", df.shape)
print(df.head(10),df.tail(10))
print(df.info())
print(df.duplicated().sum())
print(df.isnull().sum())






,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice,IsCancelled,InvoiceYear,InvoiceMonth,InvoiceDay,InvoiceHour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,False,2009,12,1,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,False,2009,12,1,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,False,2009,12,1,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,False,2009,12,1,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,False,2009,12,1,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60,False,2011,12,9,12
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60,False,2011,12,9,12
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60,False,2011,12,9,12
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680,France,14.85,False,2011,12,9,12


In [ ]:
# Drop rows where Customer ID is missing (for customer-based analysis)
df = df.dropna(subset=['Customer ID'])

# Fix Customer ID data type
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)

# Fix missing descriptions using StockCode
desc_map = (
    df.dropna(subset=['Description'])
      .drop_duplicates(subset=['StockCode'])
      .set_index('StockCode')['Description']
      .to_dict()
)

df['Description'] = df.apply(
    lambda row: desc_map.get(row['StockCode'], 'UNKNOWN') if pd.isna(row['Description']) else row['Description'],
    axis=1
)

# Remove duplicates
df = df.drop_duplicates()

# Add total price column
df['TotalPrice'] = df['Quantity'] * df['Price']

# Mark cancellations
df['IsCancelled'] = df['Invoice'].astype(str).str.startswith('C')

# Filter out bad rows (negative qty that are not cancellations)
mask_invalid = (df['Quantity'] < 0) & (~df['IsCancelled'])
df = df[~mask_invalid]

# Remove rows with zero or negative price
df = df[df['Price'] > 0]

# Extract date/time features
df['InvoiceYear'] = df['InvoiceDate'].dt.year
df['InvoiceMonth'] = df['InvoiceDate'].dt.month
df['InvoiceDay'] = df['InvoiceDate'].dt.day
df['InvoiceHour'] = df['InvoiceDate'].dt.hour

print("✅ Cleaned dataset shape:", df.shape)


✅ Cleaned dataset shape: (797815, 14)


In [ ]:
print("🧾 Data Cleaning Summary")
print("---------------------------")
print(f"Total Rows: {len(df):,}")
print(f"Unique Customers: {df['Customer ID'].nunique():,}")
print(f"Unique Products: {df['StockCode'].nunique():,}")
print(f"Countries: {df['Country'].nunique():,}")
print(f"Total Sales Value: £{df['TotalPrice'].sum():,.2f}")
print(f"Cancelled Orders: {df['IsCancelled'].sum():,}")


🧾 Data Cleaning Summary
---------------------------
Total Rows: 797,815
Unique Customers: 5,939
Unique Products: 4,646
Countries: 41
Total Sales Value: £16,289,991.29
Cancelled Orders: 18,390


In [ ]:
import pandas as pd
camp_df = pd.read_csv("KAG_conversion_data.csv")
camp_df

,ad_id,xyz_campaign_id,fb_campaign_id,age,gender,interest,Impressions,Clicks,Spent,Total_Conversion,Approved_Conversion
0,708746,916,103916,30-34,M,15,7350,1,1.430000,2,1
1,708749,916,103917,30-34,M,16,17861,2,1.820000,2,0
2,708771,916,103920,30-34,M,20,693,0,0.000000,1,0
3,708815,916,103928,30-34,M,28,4259,1,1.250000,1,0
4,708818,916,103928,30-34,M,28,4133,1,1.290000,1,1
...,...,...,...,...,...,...,...,...,...,...,...
1138,1314410,1178,179977,45-49,F,109,1129773,252,358.189997,13,2
1139,1314411,1178,179978,45-49,F,110,637549,120,173.880003,3,0
1140,1314412,1178,179979,45-49,F,111,151531,28,40.289999,2,0
1141,1314414,1178,179981,45-49,F,113,790253,135,198.710001,8,2


In [ ]:
# ✅ FINAL DATA CLEANLINESS CHECK for camp_df

import pandas as pd
import numpy as np

# 1️⃣ Basic Info
print("🔹 Dataset Shape:", camp_df.shape)
print("\n🔹 Data Types:\n", camp_df.dtypes)

# 2️⃣ Missing Values Check
print("\n🔹 Missing Values per Column:\n", camp_df.isnull().sum())

# 3️⃣ Duplicate Rows Check
print("\n🔹 Duplicate Rows:", camp_df.duplicated().sum())

# 4️⃣ Object Columns Check (non-numeric)
obj_cols = camp_df.select_dtypes(include='object').columns
print("\n🔹 Object Columns (non-numeric):", list(obj_cols))

# 5️⃣ Numeric sanity check (no negatives in key metrics)
num_cols = ['Impressions', 'Clicks', 'Spent', 'Total_Conversion', 'Approved_Conversion']
invalid_values = camp_df[num_cols][(camp_df[num_cols] < 0).any(axis=1)]
print("\n🔹 Negative Values Found:\n", invalid_values if not invalid_values.empty else "✅ None found")

# 6️⃣ Logical consistency: Approved_Conversion ≤ Total_Conversion
invalid_conv = camp_df[camp_df['Approved_Conversion'] > camp_df['Total_Conversion']]
print("\n🔹 Invalid Conversion Values:\n", invalid_conv if not invalid_conv.empty else "✅ None found")

# 7️⃣ Summary Statistics Overview
print("\n🔹 Summary Statistics:\n", camp_df.describe(include='all'))

# 8️⃣ Final Verdict
if (camp_df.isnull().sum().sum() == 0) and (camp_df.duplicated().sum() == 0) and invalid_values.empty and invalid_conv.empty:
    print("\n✅✅ DATA IS 100% CLEAN AND READY FOR ANALYSIS ✅✅")
else:
    print("\n⚠️ Data still needs cleaning — check above issues.")


🔹 Dataset Shape: (1143, 11)

🔹 Data Types:
 ad_id                    int64
xyz_campaign_id          int64
fb_campaign_id           int64
age                     object
gender                  object
interest                 int64
Impressions              int64
Clicks                   int64
Spent                  float64
Total_Conversion         int64
Approved_Conversion      int64
dtype: object

🔹 Missing Values per Column:
 ad_id                  0
xyz_campaign_id        0
fb_campaign_id         0
age                    0
gender                 0
interest               0
Impressions            0
Clicks                 0
Spent                  0
Total_Conversion       0
Approved_Conversion    0
dtype: int64

🔹 Duplicate Rows: 0

🔹 Object Columns (non-numeric): ['age', 'gender']

🔹 Negative Values Found:
 ✅ None found

🔹 Invalid Conversion Values:
 ✅ None found

🔹 Summary Statistics:
                ad_id  xyz_campaign_id  fb_campaign_id    age gender  \
count   1.143000e+03      1143.

In [ ]:
camp_df.to_csv("cleaned_KAG_conversion_data.csv", index=False)

print("✅ Data saved to 'cleaned_KAG_conversion_data.csv'")

✅ Data saved to 'cleaned_KAG_conversion_data.csv'


In [2]:
   %cd /content/data

import pandas as pd
df = pd.read_csv("cleaned_online_retail_2009_2011.csv")
df

/content/data


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice,IsCancelled,InvoiceYear,InvoiceMonth,InvoiceDay,InvoiceHour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,False,2009,12,1,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,False,2009,12,1,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,False,2009,12,1,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,False,2009,12,1,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,False,2009,12,1,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797810,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60,False,2011,12,9,12
797811,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60,False,2011,12,9,12
797812,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60,False,2011,12,9,12
797813,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680,France,14.85,False,2011,12,9,12
